# Etapa 1 - Seleção do Escopo do problema

## Imports
* Imports das bibliotecas usadas durante o código.
* Import da Base de Dados - PNS 2019
----

In [1]:
from builtins import print

import pandas as pd
from IPython.display import display
from load import Atributos_Categoricos, Atributos_Numericos
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler


In [2]:
path = r'../Arquivos Csv\Base Completa\pns2019.csv'

df = pd.read_csv(path)

df.shape

(293726, 1087)

## Definição do Escopo
O escopo tem 3 regras de delimitação:
   * Verficar se o registro teve a entrevista realizada, verficado pela variavel `V0015`:
   ``` python
   df = df[df['V0015'] == 1]
   ```
   * Delimitar a idade para 30 e 59, verificado pela `C008`:
   ``` python
   df = df[df['C008'].beetwen(30,59)]
   ```
   * Selecionar apenas as pessoas que responderam se foram **diagnosticadas ou não** com Asma, verificado pela variavel `Q074`:
   ``` python
   df = df[(df['Q074']==1) | (df['Q074']==2)]
   ```
----

In [3]:
# Tipo de entrevista - apenas Realizada == 01
# apenas entre 30 e 59 anos
# Apenas quem respondeu Q074 - se possui asma ou não

df = df[df['V0015'] == 1]

print(df['V0015'].value_counts())

df = df[df['C008'].between(30, 59)]

print(df['C008'].value_counts(ascending=True))

print(df.shape)

df = df[((df['Q074'] == 1) | (df['Q074'] == 2))]

print(df['Q074'].value_counts())

cols = ["Q00201","Q03001","Q060","Q06306","Q068","Q079","Q088","Q092","Q11006","Q11604","Q120","Q128","Q124","Q084"]

df_asmaticos = df[df['Q074'] == 1]
df_saudaveis_raw = df[df['Q074'] == 2]

# Refinar saudáveis (apenas quem respondeu 2 para todas as colunas da lista 'cols')
mask_sem_outras_doencas = (df_saudaveis_raw[cols] == 2).all(axis=1)
df_saudaveis_final = df_saudaveis_raw[mask_sem_outras_doencas]

# Unir os grupos novamente
df_final = pd.concat([df_asmaticos, df_saudaveis_final])

print(f"Asmáticos: {df_asmaticos.shape[0]}")
print(f"Saudáveis: {df_saudaveis_final.shape[0]}")

V0015
1    279382
Name: count, dtype: int64
C008
58.0    3067
57.0    3091
59.0    3093
56.0    3340
54.0    3486
53.0    3490
47.0    3495
51.0    3499
52.0    3587
55.0    3592
50.0    3625
48.0    3721
49.0    3726
46.0    3777
45.0    3811
44.0    3859
41.0    3905
43.0    3921
31.0    4029
42.0    4036
32.0    4065
33.0    4086
34.0    4088
30.0    4153
35.0    4155
38.0    4244
39.0    4264
40.0    4287
36.0    4329
37.0    4460
Name: count, dtype: int64
(114281, 1087)
Q074
2.0    48033
1.0     2376
Name: count, dtype: int64
Asmáticos: 2376
Saudáveis: 19664


In [4]:
df_final.to_csv('../Arquivos Csv/Etapa1.csv')

### N° de registros após delimitação do Escopo

* Apenas entrevistas realizadas: **279382 registros**.
* Apenas entrevistas realizas e entre 30 - 59 anos: **114281 registros**.
* Apenas entrevistas realizadas, entre 30 - 59 anos e responderam se possuem asma ou não: **22.040 registros**.
----

# Etapa 2/3 -Sseleção Conceitual de Atributos

## Rodar Etapa 2

In [5]:
df = pd.read_csv('../Arquivos Csv/Etapa1.csv')

print(df.shape)

(22040, 1088)


## Seleção dos Atributos do Modelo Conceitual
* Foram selecianados apenas as colunas que eram de interesse do modelo.
* Todas as colunas estão no arquivo `load.py`, separadas por categóricas e numéricas, além de uma breve descrição de seu respctivo significado.
* Antes: 1087 Colunas.
* Depois: 119 Colunas.
----

In [7]:
Atributos = Atributos_Categoricos + Atributos_Numericos

df = df[Atributos]

display(df.head())

print(df.shape)

df_copy = df.copy()

df_copy.to_csv(r'../Arquivos Csv/Etapa2_3.csv')

,P038,E001,E002,E003,E011,E022,Q074,J01802,J023,J024,...,P023,P02501,P02602,P02801,P029,P03202,A02305,A02306,A02307,C008
0,1.0,1.0,NaN,NaN,1.0,NaN,1.0,NaN,NaN,NaN,...,0.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,33.0
1,2.0,1.0,NaN,NaN,1.0,NaN,1.0,NaN,NaN,NaN,...,7.0,3.0,0.0,NaN,1.0,NaN,0.0,1.0,0.0,50.0
2,NaN,2.0,2.0,2.0,NaN,2.0,1.0,NaN,1.0,2.0,...,4.0,2.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,38.0
3,2.0,1.0,NaN,NaN,1.0,NaN,1.0,NaN,2.0,1.0,...,0.0,0.0,0.0,NaN,NaN,NaN,1.0,2.0,0.0,49.0
4,1.0,1.0,NaN,NaN,1.0,NaN,1.0,NaN,NaN,NaN,...,7.0,0.0,0.0,1.0,10.0,10.0,0.0,1.0,0.0,47.0


(22040, 119)


# Etapa 4 - Análise de Outliers

## Rodar Etapa 4

In [9]:
df = pd.read_csv(r'../Arquivos Csv/Etapa2_3.csv')

## Remoção de Variáveis com altos Valores nulos
* Foram selecionados todas as colunas selecionadas no modelo com mais de 80% de dados vazios.
* Em cima dessas colunas foi feito uma analise conceitual, visando analisar o contexto para os dados vazios da coluna e sua importancia do para o modelo.
* Em algumas das colunas foi verificada que apenas **uma porção dos 22.040 registros** deviam responder ela, derivada de uma resposta de outra pergunta, para essas variaveis foi feito uma analise em cima desse **novo valor máximo** ( o numero de respostas da outra pergunta) em comparação ao numero de registros que responderam de fato, caso fosse abaixo de 80% seria mantida, caso contrário, seria deletada.
* Após essa seleção o numero de atributos desceu de **119 para 87**.
* Foram eliminado os seguintes atributos: `['J01802', 'J023', 'J024', 'L01701', 'R002010', 'R028', 'R029', 'P05401', 'P05404', 'P05407', 'P05410', 'P05413', 'P05416', 'P05419', 'P05402', 'P05403', 'P05405', 'P05406', 'P05408', 'P05409', 'P05411', 'P05412', 'P05414', 'P05415', 'P05417', 'P05418', 'P05421', 'P05422']`.
----

In [10]:

mask = (df.isna().sum() / len(df)) > 0.80

colunas_ruins = df.columns[mask].tolist()

colunas_boas = df.columns[~mask].tolist()

print(f"Colunas para deletar: {colunas_ruins}")

colunas_protegidas = ['P051','P04401','P04405','P04406','A01403']

colunas_ruins = [col for col in colunas_ruins if col not in colunas_protegidas]

df = df.drop(columns=colunas_ruins+['P03202','R026','R010','R025','VDE001'])

print(f"Colunas De fato Removidas: {colunas_ruins}")

print(df.shape)

Colunas para deletar: ['J01802', 'J023', 'J024', 'L01701', 'R002010', 'R028', 'R029', 'A01403', 'P051', 'P04401', 'P04405', 'P04406', 'P05401', 'P05404', 'P05407', 'P05410', 'P05413', 'P05416', 'P05419', 'P05402', 'P05403', 'P05405', 'P05406', 'P05408', 'P05409', 'P05411', 'P05412', 'P05414', 'P05415', 'P05417', 'P05418', 'P05421', 'P05422']
Colunas De fato Removidas: ['J01802', 'J023', 'J024', 'L01701', 'R002010', 'R028', 'R029', 'P05401', 'P05404', 'P05407', 'P05410', 'P05413', 'P05416', 'P05419', 'P05402', 'P05403', 'P05405', 'P05406', 'P05408', 'P05409', 'P05411', 'P05412', 'P05414', 'P05415', 'P05417', 'P05418', 'P05421', 'P05422']
(22040, 87)


## Remoção de Registros de Atributos com Valores Constantes

* Foi feito uma análise para encontrar atributos categóricos com um valor predominante (Acima de 80% das respostas).
* Uma vez identificado foi feita uma remoção dos registros que compunham as outras respostas ( fora dos 80% ou mais).
* Após deixar apenas os registros com as outras respostas, este atributo se torna uma constante, portanto, ele é eliminado.
* Foram eliminado os seguintes atributos: `['A00601','VDE002','F007011']`.
----

In [11]:
cols_Constantes = ['A00601','VDE002','F007011']

df.drop(columns=cols_Constantes,inplace=True)
df.shape

(22040, 84)

## Remoção de Valores Desinformativos
* Valores desinformativos é considerados respostas que não trazem nenhuma informação relevante, como: 'não se aplica', 'não sei', etc.
---

In [12]:
print(df.columns)
print(df.shape)
# Lista de colunas e seus valores alvo para remoção
colunas_remover = {
    'Q09202': 4,
    'C009':9
}

# Cria a máscara para todas as condições de uma vez
mask = df.isin({'Q09201': [3], 'Q09202': [4]}).any(axis=1)

# Filtra
df = df[~mask]

print(df.shape)

Index(['Unnamed: 0', 'P038', 'E001', 'E002', 'E003', 'E011', 'E022', 'Q074',
       'V0001', 'A005010', 'A005012', 'A01403', 'A01501', 'A016010', 'A01901',
       'D001', 'VDE014', 'M01001', 'P02101', 'P051', 'P034', 'P039', 'P044',
       'P04401', 'P04405', 'P04406', 'P02102', 'P02401', 'P03201', 'M011031',
       'M009', 'M011011', 'M011051', 'M011071', 'A02201', 'C009', 'C006',
       'J01101', 'N00101', 'N012', 'N016', 'N017', 'N018', 'A009010',
       'VDD004A', 'P050', 'P052', 'P067', 'P06701', 'P068', 'P04501', 'P04502',
       'P02601', 'P027', 'VDF003', 'P03701', 'P03702', 'P00104', 'P00404',
       'J012', 'P035', 'P03904', 'P03905', 'P03906', 'P006', 'P00901',
       'P01001', 'P01101', 'P013', 'P015', 'P02001', 'P01601', 'P018', 'P019',
       'P02002', 'P023', 'P02501', 'P02602', 'P02801', 'P029', 'A02305',
       'A02306', 'A02307', 'C008'],
      dtype='object')
(22040, 84)
(22040, 84)


## Remoção de Outliers

* Será feito uma análise de outliers em valores númericos, contudo, atributos relevantes (peso, altura) não serão removidos outliers por meio do IQR ou do desvio-padrão, apenas conceitualmente (valores ilógicos), sendo mantido outliers lógicos.
----

In [13]:
# Cols_Numéricas = ['J012' - N° consultas medico, 'P03905' - Andar horas,'P029' N° doses em media,'P03202' N° numa ocasiao, 'VDF003'- Renda per Capita

print('Asmaticos: Valores Válidos')
print(df[(df['Q074'] == 1) & (df['J012'] <= 10)]['J012'].value_counts(ascending=True))


print('Saudaveis: Valores Válidos')
print(df[(df['Q074'] == 2) & (df['J012'] <= 10)]['J012'].value_counts(ascending=True))


Asmaticos: Valores Válidos
J012
9.0       5
7.0      32
8.0      63
10.0    156
5.0     156
6.0     161
4.0     211
3.0     325
1.0     383
2.0     417
Name: count, dtype: int64
Saudaveis: Valores Válidos
J012
9.0       45
7.0      112
8.0      153
10.0     318
6.0      401
5.0      713
4.0      964
3.0     1982
2.0     3585
1.0     5258
Name: count, dtype: int64


* A classe Asmática empiricamente possui um bom corte em 10, totalizando: **1.435 registros (mais os asmáticos que possuem respostas `nan`) de 1.544, cerca de 94.2%** dos dados de asmáticos.
    * **Redução de 5.6%**.
* A classe Saudavel ,por sua vez, com este corte totalizando: **11.387 registros de 11.574 (mais os saudáveis que possuem respostas `nan`), cerca de 98.4%** dos dados de saudáveis.
    * **Redução de 1.6%**.

In [14]:
mask_remover = ((df['J012'] <= 10) | (df['J012'].isna()))

df = df[mask_remover]

print(df.shape)

print(f'Classes: {df['Q074'].value_counts()}' )

(21658, 84)
Classes: Q074
2.0    19428
1.0     2230
Name: count, dtype: int64


In [15]:
print('Asmaticos:')
print(df[(df['Q074'] == 1)]['P03905'].value_counts())


print('Saudaveis:')
print(df[(df['Q074'] == 2)]['P03905'].value_counts())

Asmaticos:
P03905
1.0     109
2.0     108
4.0     105
0.0     101
3.0      92
8.0      92
5.0      68
6.0      61
7.0      25
10.0     12
12.0     10
9.0       9
11.0      2
13.0      1
Name: count, dtype: int64
Saudaveis:
P03905
4.0     1000
2.0      997
8.0      923
1.0      800
0.0      773
3.0      765
6.0      712
5.0      631
7.0      276
10.0     104
12.0      66
9.0       61
11.0       7
15.0       2
14.0       2
16.0       1
13.0       1
Name: count, dtype: int64


* Distribuição não normal
* Uso do IQR.

In [16]:
# Cálculo dos quartis
Q1 = df['P03905'].quantile(0.25)
Q3 = df['P03905'].quantile(0.75)

# Cálculo do IQR
IQR = Q3 - Q1

# Definição dos limites superior e inferior
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

# Máscara para manter apenas valores validos e valores nulos
mask_iqr = (df['P03905'] >= limite_inferior) & (df['P03905'] <= limite_superior) | (df['P03905'].isna())

# Aplicação do corte no DataFrame
df = df[mask_iqr]

print(df['Q074'].value_counts())
print(df.shape)

Q074
2.0    19422
1.0     2229
Name: count, dtype: int64
(21651, 84)


* Distribuição não normal
* Uso do IQR.

In [17]:
# Cálculo dos quartis
Q1 = df['P029'].quantile(0.25)
Q3 = df['P029'].quantile(0.75)

# Cálculo do IQR
IQR = Q3 - Q1

# Definição dos limites superior e inferior
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

# Máscara para manter apenas inliers e valores nulos
mask_iqr = (df['P029'] >= limite_inferior) & (df['P029'] <= limite_superior) | (df['P029'].isna())

# Aplicação do corte no DataFrame
df = df[mask_iqr]

print(df['Q074'].value_counts())
print(df.shape)

Q074
2.0    19081
1.0     2189
Name: count, dtype: int64
(21270, 84)


In [18]:
df_copy = df.copy()

df_copy.to_csv(r'../Arquivos Csv/Etapa4.csv')

# Etapa 5 - Tratamento de Dados Ausentes e Vazios

## Rodar Etapa 5

In [19]:
df = pd.read_csv(r'../Arquivos Csv/Etapa4.csv')

## Dados Vazios

* Serão eliminados, todas os atributos são pouco relevantes e não há forma de imputação ou de discretização eficiente.

In [20]:
Cols_drop = ['P03905','P03906','P03201']

df.drop(inplace=True,columns=Cols_drop)

df.shape


(21270, 82)

## Dados Ausentes - Imputação / discretização

* Para os atributos que possuem dados ausentes será implementado

In [21]:
print('Asmaticos:')
print(df[(df['Q074'] == 1)]['VDE014'].isna().sum())


print('Saudaveis:')
print(df[(df['Q074'] == 2)]['VDE014'].isna().sum())

print(df[(df['E001'] == 2) & (df['E002'] == 2) & (df['E003'] == 2)]['VDE014'].isna().sum())

print(df['VDE014'].isna().sum())


Asmaticos:
595
Saudaveis:
4012
4607
4607


* Todos os registros que responderam que não estao em um emprego remunerado em: `E001`, `E002`, `E003`. Estão como `NaN` em `VDE014`, portanto, será imputado valor 0, equivalente a desempregado.

In [22]:
mask_imputar = (
    (df['E001'] == 2) &
    (df['E002'] == 2) &
    (df['E003'] == 2) &
    (df['VDE014'].isna())
)

df.loc[mask_imputar, 'VDE014'] = 0

print(df['VDE014'].value_counts(dropna=False))


VDE014
0.0     4607
4.0     2790
9.0     2291
1.0     1961
7.0     1712
2.0     1656
11.0    1315
3.0     1238
8.0     1190
5.0      876
6.0      860
10.0     769
12.0       5
Name: count, dtype: int64


* Como as colunas `E001`, `E002`, `E003` representa as pessoas não remuneradas e são o resultado do valor 0 na variavel `VDE014`, elas serão removidas apos a transformação.

* Valor do salário minímo em 2019 era de  R$ 998,00.

In [23]:
Salario_min = 998.0

# 1. Definimos os limites (bins)
bins = [
    0, #1
    0.25 * Salario_min, # 2
    0.5 * Salario_min, # 3
    1 * Salario_min, # 4
    2 * Salario_min, # 5
    3 * Salario_min, # 6
    5 * Salario_min, # 7
    np.inf
]

# 2. Definimos os labels como números de 1 a 7
labels = [1, 2, 3, 4, 5, 6, 7]

# 3. Substituímos a coluna original
# AQUI ESTÁ A CORREÇÃO:
# Trocamos '.astype(int)' por '.astype("Int64")'
# O tipo "Int64" (com 'I' maiúsculo) do Pandas permite ter números inteiros E valores NaN na mesma coluna,
# sem disparar o ValueError.
df['Salario_min'] = pd.cut(df['VDF003'], bins=bins, labels=labels, include_lowest=True).astype("Int64")

df.drop(inplace=True, columns=['VDF003'])

# Verificando a substituição
print(df['Salario_min'].value_counts(dropna=False).sort_index())

Salario_min
1       2174
2       3500
3       5604
4       5402
5       1934
6       1394
7       1252
<NA>      10
Name: count, dtype: Int64


* IMC = Peso / Altura^2


In [24]:
print(df['P00104'].isna().sum())

print(df['P00404'].isna().sum())

df.dropna(subset=['P00104','P00404'],inplace=True)

# Cálculo do IMC
# Assumindo que P00404 (Altura) está em centímetros, dividimos por 100
df['IMC'] = df['P00104'] / ((df['P00404'] / 100) ** 2)

# Opcional: Remover possíveis NaNs gerados pelo cálculo (caso peso ou altura fossem nulos)
df = df.dropna(subset=['IMC'])

# Visualizando as primeiras linhas do cálculo
print(df[['P00104', 'P00404', 'IMC']].head())

bins = [0, 18.5, 25, 30, 35, 40, np.inf]
labels = [1, 2, 3, 4, 5, 6]

# Usamos right=False para que o valor exato (ex: 25.0) pule para a próxima categoria
df['IMC'] = pd.cut(df['IMC'], bins=bins, labels=labels, right=False).astype(int)

print(df['IMC'].value_counts().sort_index())

df.drop(columns=['P00104','P00404'],inplace=True)

186
186
   P00104  P00404        IMC
0    72.0   169.0  25.209201
1    78.0   165.0  28.650138
3    65.0   156.0  26.709402
4    88.0   163.0  33.121307
5    77.0   176.0  24.857955
IMC
1     286
2    8409
3    8378
4    3085
5     745
6     181
Name: count, dtype: int64


* Para a variavel `P039` e `P03904` será feito um Concatenação Lógica.

In [25]:
mask = (
    df['E011'].between(1,3)  # 1 a 3 são as unicas respostas de  E011
)
print(df['P038'].count())
df[mask]['P038'].value_counts()

16551


P038
2.0    10185
1.0     6366
Name: count, dtype: int64

* A variavel `E011` se refer as pessoas ocupada (empregada), logo, quem não está ocupado não realiza atividades fisicas por conta do emprego.
* Portanto será imputado valor 0 nesses registros.
* Caso a resposta de `P038` e `P039` sejam 2, ou seja, Não, será imputado 0 na variavel `P03904`, pois a pessoa é empregada mas não realiza exercicio devido ao emprego.
* Para a variavel `P03904`, zero é igual a não realiza esforço por conta do trablho.
* Como todas as informações estão contidas (intensidade e se realiza ou não) na variavel `P03904`, as variaveis `P038` e `P039`, serão excluidas.

In [26]:
# 1. Imputar nulos de P038 como 0 (Não possui emprego/Não se aplica)
df['P038'] = df['P038'].fillna(0)

display(df['P038'].value_counts())

def consolidar_esforco(row):
    # Se ambos responderam 'Não' (2.0)
    if row['P038'] == 2 and row['P039'] == 2:
        return 0
    # Se o valor de P03904 for nulo, mas houve resposta 1 em P038 ou P039,
    # geralmente imputamos 0 ou mantemos NaN conforme a sua regra.
    # Aqui, garantimos que se o fluxo diz que não teve esforço, é 0.
    return row['P03904']

# Aplicando a lógica
df['P03904'] = df.apply(consolidar_esforco, axis=1)

# 3. Limpeza final: tratar nulos restantes em P03904 como 0
# (Ex: pessoas que não trabalham ou não passaram pelo filtro)
df['Exercício pesado trabalho'] = df['P03904'].fillna(0).astype(int)

# Verificação
print(df['Exercício pesado trabalho'].value_counts().sort_index())

df.drop(inplace=True, columns=['P038','P039','P03904'])

P038
2.0    10185
1.0     6366
0.0     4533
Name: count, dtype: int64

Exercício pesado trabalho
0    13404
1      521
2      683
3      975
4      553
5     2558
6     1703
7      687
Name: count, dtype: int64


C:\Users\Enzo Gripp\AppData\Local\Temp\ipykernel_25020\312423121.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Exercício pesado trabalho'] = df['P03904'].fillna(0).astype(int)


* A variável `P050`, `P051` e `P052`, possuem a mesma lógica de salto, sendo a `P050` a pergunta originária.
* Caso a resposta de `P050` seja 2 essa pessoa é redirecionada para `P051`.
* Caso a resposta de `P050` seja igual a 3 a pessoa é refirecionada para `P052`.
* A váriavel `M01001` e `P058` (em domicilio) informa se a pessoa se enquadra em fumantes passivos, esses registros dependem que a resposta da variavel `M009` seja 1 ou 3 (trabalho em ambientes fechados).
* A classificação de fumante passivel será aplicada apenas se o registro em questão não usar ou ter usado algum produto de tabaco.
---
* As 4 váriaveis dizem se a pessoa fuma atualmente (diariamente ou não), se já fumou no passado (diaramente ou não) ou se se enquadra em fumante passivo. Portanto, as 4 serão agrupas em uma variavel catégorica.
* Serão as seguintes categórias:
	* Fumante diário (Resposta da `P050` == 1)
	* Fumante ocasional (Resposta da `P050` == 2 e Resposta da `P051` == 2)
    * Fumante ocasional 'ex-diario' (``P050 ``== 2 e ``P051`` == 1)
	* Ex-Fumante diário (Resposta da `P050` == 3 e Resposta da `P052` == 1 )
	* Ex-Fumante ocasional (Resposta da `P050` == 3 e Resposta da `P052` == 2)
	* Fumante passivo (Resposta da `P050` == 3 e Resposta da `P052` == 3  e Resposta da `M01001` == 1) ou (Resposta da `P050` == 3 e Resposta da `P052` == 3  e Resposta da `P068` == 1 ou 2).
	* Não fumante (Resposta da `P050` == 3 e Resposta da `P052` == 3 e  e Resposta da `M01001` == 2)

In [27]:
print(df.shape)

# 2. Definir as condições lógicas baseadas na Opção 2
condicoes = [
    # Fumante diário
    (df['P050'] == 1),

    # Fumante ocasional (Sempre foi ocasional)
    (df['P050'] == 2) & (df['P051'] == 2),

    # Fumante ocasional (ex-diário - Reduziu o uso)
    (df['P050'] == 2) & (df['P051'] == 1),

    # Ex-Fumante diário (Parou totalmente)
    (df['P050'] == 3) & (df['P052'] == 1),

    # Ex-Fumante ocasional (Parou totalmente)
    (df['P050'] == 3) & (df['P052'] == 2),

    # Fumante passivo (Nunca fumou, mas foi exposto no trabalho)
    (df['P050'] == 3) & (df['P052'] == 3) & (df['M01001'] == 1) |
    (df['P050'] == 3) & (df['P052'] == 3) & (df['P068'].between(1,2)),

    # Não fumante (Nunca fumou e não foi exposto ou não trabalha em local fechado)
    (df['P050'] == 3) & (df['P052'] == 3) & ((df['M01001'] == 2) | (df['M009'] == 2) | (df['VDE014']==0) )
]

# Mapeamento de 'perfil_tabagismo':
# 1 = Fumante diário
# 2 = Fumante ocasional
# 3 = Fumante ocasional (ex-diário)
# 4 = Ex-Fumante diário
# 5 = Ex-Fumante ocasional
# 6 = Fumante passivo
# 7 = Não fumante
# 0 = Nao Classificado / Dados Inconsistentes (Default)
categorias = [1, 2, 3, 4, 5, 6, 7]

# 4. Criar a nova variável numérica no df
df['perfil_tabagismo'] = np.select(condicoes, categorias, default=0).astype(int)

df.drop(inplace=True, columns=['P050','P051','P052','M01001','M009','P067','P06701','P068'])

# Visualizar o resultado para conferência dos códigos numéricos
print(df['perfil_tabagismo'].value_counts())
print(df['perfil_tabagismo'].value_counts().sum())
print(df.shape)

(21084, 79)
perfil_tabagismo
7    13254
4     3627
1     2224
6     1116
5      607
2      131
3      125
Name: count, dtype: int64
21084
(21084, 72)


* A variavel `A005010` se refere forma de abastecimento da casa, sendo 6 catégorias.
* A variavel `A005012` se refere caso ela não tenha respondido que o meio de abastacimento seja a Rede geral de distribuição, pergunta se ela está ligada a Rede geral de distribuição
---
* As 2 variavel serão juntadas em 1, onde haverá 2 categórias, está ligada ou não a rede geral de distribuição.
    * Ligado a rede geral de distruição (Resposta da `A005010` == 1 ou Resposta da `A005010` == `[2,3,4,5,6]` e Resposta da `A005012` == 1)
    * Não ligado a rede geral de distruição (Resposta da `A005010` == `[2,3,4,5,6]` e Resposta da `A005012` == 2)



In [28]:
print(df.shape)

condicoes = [
    # Ligado a rede
    (df['A005010']==1) |
    ((df['A005010'].isin([2,3,4,5,6])) & (df['A005012']==1)),

    # Não ligado a rede
    (df['A005010'].isin([2,3,4,5,6])) & (df['A005012']==2)
]

# Mapeamento de 'acesso_rede_agua':
# 1 = Ligado a rede geral de distribuição
# 2 = Não ligado a rede geral de distribuição
# 0 = Nao Classificado / Dados Inconsistentes (Default)
categorias = [1, 2]

df['acesso_rede_agua'] = np.select(condicoes, categorias, default=0).astype(int)

print(df['acesso_rede_agua'].value_counts(dropna=False))

df.drop(inplace=True, columns=['A005010', 'A005012'])

print(df.shape)

(21084, 72)
acesso_rede_agua
1    17311
2     3773
Name: count, dtype: int64
(21084, 71)


* As variaveis `P044`, `P04401` ,`P04405`, `P04406` se referem a realização de exercicios fisicos devido a atividades habituais.
---
* As 3 variaveis serão juntadas em uma variável continua q será discretizada.
* Primeiro será imputado 0 para os registros que responderam 2 em `P044` (não realizam atividades domésticas pesadas)
* Após a imputação será feito uma soma das horas + min gastos nas atividades domésticas (resultado em minutos). O valor resultante será multiplicado pelo numero de dias e depois divido por 7 (média da semana).
* Por fim será feito a separação em tercis dos dados (pouco tempo, comun tempo, muito tempo).

In [29]:
print(df.shape)
# 1. Garantir que valores nulos nas colunas de tempo sejam tratados como 0
# (Isso evita erros matemáticos na fórmula)
df['P04405'] = df['P04405'].fillna(0)
df['P04406'] = df['P04406'].fillna(0)
df['P04401'] = df['P04401'].fillna(0)

# 2. Calcular a média diária em minutos (Sua nova lógica)
# Se P044 == 2, recebe 0. Caso contrário: ((Horas * 60) + Minutos) * Dias / 7
df['media_diaria_domestico_min'] = np.where(
    df['P044'] == 2,
    0,
    (((df['P04405'] * 60) + df['P04406']) * df['P04401']) / 7
)

# 3. Separar em tercis APENAS quem tem média > 0
mask_ativos = df['media_diaria_domestico_min'] > 0

# O pd.qcut fará a divisão em 3 partes iguais (tercis) baseada na distribuição real de quem faz faxina
tercis_ativos = pd.qcut(
    df.loc[mask_ativos, 'media_diaria_domestico_min'],
    q=3,
    labels=['Pouco tempo', 'Tempo comum', 'Muito tempo']
)

# 4. Consolidar na variável categórica final
# Inicializa todos com 'Nenhum esforco' (cobre a imputação de 0 do passo 2)
df['Exercicio_domestico'] = 'Nenhum esforco'

# Substitui os registros > 0 pelos rótulos dos tercis
df.loc[mask_ativos, 'Exercicio_domestico'] = tercis_ativos

# Tratamento para casos onde o entrevistado não respondeu se faz ou não faxina (missing data na pergunta base)
df.loc[df['P044'].isna(), 'Exercicio_domestico'] = 'Dados Inconsistentes'

df.drop(inplace=True, columns=['P04405','P04406','P04401','P044'])

# Para visualizar os limites reais de tempo (em minutos) que o pandas usou para cortar os tercis:
limites_tercis = pd.qcut(df.loc[mask_ativos, 'media_diaria_domestico_min'], q=3).value_counts().sort_index()
print("Distribuição das categorias:")
print(df['Exercicio_domestico'].value_counts())
print("\nLimites numéricos dos tercis (em minutos/dia):")
print(limites_tercis)

(21084, 71)
Distribuição das categorias:
Exercicio_domestico
Nenhum esforco    17817
Pouco tempo        1456
Muito tempo        1060
Tempo comum         751
Name: count, dtype: int64

Limites numéricos dos tercis (em minutos/dia):
media_diaria_domestico_min
(0.142, 25.714]     1456
(25.714, 42.857]     751
(42.857, 780.0]     1060
Name: count, dtype: int64


* As variaveis `A02201`, `A02305`, `A02306` e `A02307` se referem ao registro possuir animais de estimação com pelagem ou não.
* A lógica de pulo está na `A02201`, na qual pergunta se  a pessoa possui animal de estimação ou não, caso possua ela responde quais são os animais.
---
* As 4 váriaveis serão juntadas em uma variavel, ná qual será 2 catégorias:
    * Será caso a Resposta de `A02201` == 1 e `[A02305,A02306,A02307]` > 0 (Animais que afetam a respiração).
    * Será caso a Resposta de `A02201`== 2 ou Resposta de `[A02305,A02306,A02307]` == 0 (Animais que não afetam a respiração)

In [30]:
print(df.shape)

# 1. Tratar os valores nulos gerados pelo pulo da A02201
df['A02305'] = df['A02305'].fillna(0) # Gatos
df['A02306'] = df['A02306'].fillna(0) # Cachorros
df['A02307'] = df['A02307'].fillna(0) # Aves

# 2. Definir as condições
condicoes_pets = [
    # Possui Animal com Pelo/Pena (Gatos, Cachorros ou Aves > 0)
    (df['A02201'] == 1) & ((df['A02305'] > 0) | (df['A02306'] > 0) | (df['A02307'] > 0)),

    # Não possui animais com pelo/pena (Não tem bicho nenhum OU só tem outros tipos de bichos)
    (df['A02201'] == 2) | ((df['A02201'] == 1) & (df['A02305'] == 0) & (df['A02306'] == 0) & (df['A02307'] == 0))
]

# Mapeamento de 'perfil_animal_pelo':
# 1 = Animais que afetam a respiração
# 2 = Animais que não afetam a respiração
# 0 = Nao Classificado / Dados Inconsistentes (Default)
categorias_pets = [1, 2]

# 3. Aplicar a transformação numérica
df['perfil_animal_pelo'] = np.select(
    condicoes_pets,
    categorias_pets,
    default=0
).astype(int)

# Visualizar o resultado para validação dos códigos numéricos
print(df['perfil_animal_pelo'].value_counts(dropna=False))

df.drop(inplace=True, columns=['A02201', 'A02305', 'A02306', 'A02307'])

print(df.shape)

(21084, 69)
perfil_animal_pelo
1    12235
2     8849
Name: count, dtype: int64
(21084, 66)


* A variável `J012` se refere ao N° de vezes que consultou um médico em um ano, essa variável tem uma lógica de salto da varivavel `J01101` == 1, que seria quando a pessoa foi no médico pelo ultima vez.
---
* Essa váriavel continua será discretizada em 2 grupos, sendo ignorado os valores igual a zero (que não foram ao médico no ultimo ano).
* As catégorias serão:
    * Poucas vezes: 1° Metade.
    * Muitas vezes: 2° Metade.
    * Não foram: Valores == 0.


In [31]:
print(df.shape)
# J01101 não possui missing values
df['J012'] = df['J012'].fillna(0)

# 2. Criar máscara apenas para quem foi ao médico (Valores > 0)
mask_ativos_medico = df['J012'] > 0

# 3. Calcular os tercis apenas para os ativos usando labels numéricos
# O duplicates='drop' colapsou os cortes em 2 faixas, por isso usamos 2 labels
tercis_medico = pd.qcut(
    df.loc[mask_ativos_medico, 'J012'],
    q=3,
    labels=[1, 2],
    duplicates='drop'
)

# Mapeamento de 'frequencia_medico':
# 0 = Nao foram
# 1 = Poucas vezes
# 2 = Muitas vezes
# 4. Criar a coluna final numérica (Inicializa com 0 para cobrir quem não foi)
df['frequencia_medico'] = 0

# Atualiza os registros > 0 com os códigos numéricos dos tercis
df.loc[mask_ativos_medico, 'frequencia_medico'] = tercis_medico

# Garante o tipo de dado como inteiro
df['frequencia_medico'] = df['frequencia_medico'].astype(int)

# df.drop(inplace=True,columns=['J012'])

# Visualizar a distribuição e os limites numéricos que o pandas calculou
print("Distribuição das categorias:")
print(df['frequencia_medico'].value_counts())

print("\nVerificando os cortes numéricos dos tercis:")
print(pd.qcut(df.loc[mask_ativos_medico, 'J012'], q=3, duplicates='drop').value_counts().sort_index())

(21084, 66)
Distribuição das categorias:
frequencia_medico
1    11724
0     6034
2     3326
Name: count, dtype: int64

Verificando os cortes numéricos dos tercis:
J012
(0.999, 3.0]    11724
(3.0, 10.0]      3326
Name: count, dtype: int64


* As variáveis `M011051`,`M011011`,`M011031`,`M011071` se referem a exposição de riscos a asma no trabalho.
* Os valores nulos são aqueles que não trabalham.
---
* Será agrupado as 4 variaveis em uma variavel categorica.
* Primeiro será imputado zero para os valores nulos já que são os registros que não trabalham.
* As categorias são:
    * Caso a Resposta de `[M011051,M011011,M011031,M011071]` == 1, sendo aplicado ou (Sofrem exposição no trabalho).
    * Caso a Resposta de `[M011051,M011011,M011031,M011071]` == 2, sendo aplicado e (Não Sofrem exposição no trabalho).
    * Caso a Resposta de `[M011051,M011011,M011031,M011071]` == 0, sendo aplicado e (Não trabalham).

In [32]:
# 1. Definir a lista de variáveis para facilitar o preenchimento
vars_exposicao = ['M011011', 'M011031', 'M011051', 'M011071']

# Imputar 0 para os valores nulos (que representam pessoas que não trabalham ou pularam o módulo)
for var in vars_exposicao:
    df[var] = df[var].fillna(0)

condicoes = [
    # Sofrem exposição no trabalho (Se QUALQUER UMA for igual a 1 usa-se o | OU)
    (df['M011011'] == 1) | (df['M011031'] == 1) | (df['M011051'] == 1) | (df['M011071'] == 1),

    # Não sofrem exposição no trabalho (Se TODAS forem iguais a 2 usa-se o & E)
    (df['M011011'] == 2) & (df['M011031'] == 2) & (df['M011051'] == 2) & (df['M011071'] == 2),

    # Não trabalham (Se TODAS forem iguais a 0 usa-se o & E)
    (df['M011011'] == 0) & (df['M011031'] == 0) & (df['M011051'] == 0) & (df['M011071'] == 0)
]

# Mapeamento de 'exposicao_trabalho':
# 1 = Sofrem exposicao no trabalho
# 2 = Nao sofrem exposicao no trabalho
# 3 = Nao trabalham
# -1 = Dados Inconsistentes (Default)
categorias = [1, 2, 3]

# 4. Criar a nova variável numérica consolidada
df['exposicao_trabalho'] = np.select(condicoes, categorias, default=-1).astype(int)

# Visualizar o resultado para validação dos códigos numéricos
print("Distribuição das categorias de exposição no trabalho:")
print(df['exposicao_trabalho'].value_counts(dropna=False))

# Remover as colunas originais caso não vá mais utilizá-las
df.drop(columns=vars_exposicao, inplace=True)

Distribuição das categorias de exposição no trabalho:
exposicao_trabalho
2    10373
1     6178
3     4533
Name: count, dtype: int64


* Map das Regiões de cada estado.

In [33]:
# 1. Criar o dicionário relacionando o dígito da dezena ao código numérico da região
# Mapeamento de 'regiao_br':
# 1 = Norte
# 2 = Nordeste
# 3 = Sudeste
# 4 = Sul
# 5 = Centro-Oeste
# 0 = Dados Inconsistentes / Vazio (Default)
mapa_regioes = {
    1: 1,
    2: 2,
    3: 3,
    4: 4,
    5: 5
}

# 2. Aplicar a divisão inteira (// 10) para extrair o primeiro dígito e mapear
df['regiao_br'] = (df['V0001'].fillna(0) // 10).map(mapa_regioes)

# 3. Tratar possíveis dados inconsistentes e forçar o tipo inteiro
df['regiao_br'] = df['regiao_br'].fillna(0).astype(int)

# Visualizar o resultado para conferência dos códigos numéricos
print("Distribuição das Regiões:")
print(df['regiao_br'].value_counts())

Distribuição das Regiões:
regiao_br
2    6932
3    4556
1    4259
4    2675
5    2662
Name: count, dtype: int64


* As variaveis `A01403`,`A01501` indicam o nível de acesso ao sistema de esgoto.
* `A01501` possui 83 registros vazios, que são as pessoas que responderam == 2 em `A01403`, que seria o equivalente a pessoa não ter banheiro em casa, banheiro compartilhado ou usasse um buraco no proprio terreno, ou seja, o maior nível de precáridade de acesso ao sistema de esgoto possível.
---
* Será unido os valores == 2 na de `A01403`, como uma nova categoria em `A01501`, "representadno Sem instalação sanitária / Extrema vulnerabilidade".
* Por fim será reduzido o N° de categorias em `A01501`, para 5:
    * 1: Ideal (Rede pública)
    * 2 e 3: Adequado (Fossa Séptica)
    * 4, 5 e 6: Perigoso / Inadequado (Fossa rudimentar, vala, rio)
    * 7: Outro
    * Pulo do questionário: Extrema vulnerabilidade (Sem banheiro)

In [34]:
print(df.shape)

print(df.shape)

# 1. Definir as condições lógicas (com a regra de pulo/vulnerabilidade no topo por prioridade)
condicoes_esgoto = [
    # A Regra de Pulo: Extrema Vulnerabilidade (Sem banheiro nenhum)
    (df['A01403'] == 2),

    # 1 - Ideal (Rede pública)
    (df['A01501'] == 1),

    # 2 e 3 - Adequado (Fossa Séptica)
    df['A01501'].isin([2, 3]),

    # 4, 5 e 6 - Perigoso / Inadequado (Rudimentar, Vala, Rio/Mar)
    df['A01501'].isin([4, 5, 6]),

    # 7 - Outro
    (df['A01501'] == 7)
]

# 2. Definir os códigos numéricos equivalentes para as categorias
# Mapeamento de 'condicao_saneamento':
# 5 = Extrema Vulnerabilidade (Sem banheiro)
# 1 = Ideal (Rede Publica)
# 2 = Adequado (Fossa Septica)
# 3 = Perigoso / Inadequado
# 4 = Outro
# 0 = Nao Classificado / Dados Inconsistentes (Default)
categorias_esgoto = [5, 1, 2, 3, 4]

# 3. Criar a nova variável numérica consolidada
df['condicao_saneamento'] = np.select(
    condicoes_esgoto,
    categorias_esgoto,
    default=0
).astype(int)

# Visualizar como ficou a distribuição dos códigos numéricos
print("Distribuição das condições de saneamento:")
print(df['condicao_saneamento'].value_counts(dropna=False))

# 4. Eliminar as colunas antigas e as variáveis de trabalho
df.drop(inplace=True, columns=['A01403', 'A01501', 'E001', 'E002', 'E003', 'E011', 'E022'])
print(df.shape)

(21084, 65)
(21084, 65)
Distribuição das condições de saneamento:
condicao_saneamento
1    9814
2    5960
3    5038
5     223
4      49
Name: count, dtype: int64
(21084, 59)


* As variaveis `J01101` e `J012` se referem as consultas médicas no ultimo ano.
* O registro só pode responder `J012` caso `J01101` == 1, portanto será usado apenas o N° de consultas no ultimos 12 meses (contido em `J012`).
---
* Será preenchido os valores `NaN` de `J012` com 0, pois é o N° de vezes que alguem foi no medico no utlimo ano caso a pessoa não tenha ido no medico no ultimo ano.
* Por fim será discretizado em 3 catégorias, as categorias são:
    * 1 vez: `J012` == 1.
    * 2 a 3 Vezes: `J012` == 2.
    * 4 vezes ou mais: `J012` >= 4.

In [35]:
print(df.shape)

# 1. Tratamento inicial: Garante que os NaNs (quem pulou a pergunta porque não foi ao médico) virem 0
df['J012'] = df['J012'].fillna(0)

# 2. Definir as condições lógicas
condicoes = [
    # 0 Vezes
    (df['J012'] == 0),
    # 1 Vez
    (df['J012'] == 1),
    # 2 a 3 vezes
    (df['J012'].between(2, 3)),
    # 4 vezes ou mais
    (df['J012'] >= 4)
]

# 3. Definir os códigos numéricos equivalentes para as categorias
# Mapeamento de 'num_consultas_ano':
# 0 = 0 vezes
# 1 = 1 Vez
# 2 = 2 a 3 vezes
# 3 = 4 vezes ou mais
# -1 = Dados Inconsistentes (Default)
categorias = [0, 1, 2, 3]

# 4. Criar a nova variável numérica consolidada
df['num_consultas_ano'] = np.select(condicoes, categorias, default=-1).astype(int)

# Visualizar a distribuição dos novos códigos numéricos
print("Distribuição do número de consultas no último ano:")
print(df['num_consultas_ano'].value_counts(dropna=False))

# 5. Eliminar as colunas antigas do questionário
df.drop(inplace=True, columns=['J012', 'J01101'])

print(df.shape)

(21084, 59)
Distribuição do número de consultas no último ano:
num_consultas_ano
2    6194
0    6034
1    5530
3    3326
Name: count, dtype: int64
(21084, 58)


* A variavel `VDD004A` se refere ao nível mais alto de educação do registro.
---
* Ele possui inicialmente 7 categorias, será feito uma redução dimensionalidade para 4 categorias.
* As categorias são:
    * Médio incompleto/ não feito ou equivalente: `VDD004A` entre 1 e 4
    * Médio Completo ou equivalente: `VDD004A` == 5
    * Superior Incompleto ou equivalente: `VDD004A` == 6
    * Superior completo ou equivalente: `VDD004A` == 7

In [36]:
print(df.shape)

condicoes = [
    # Médio incompleto/ não feito ou equivalente
    (df['VDD004A'].between(1,4)),
    # Médio Completo ou equivalente
    (df['VDD004A']==5),
    # Superior Incompleto ou equivalente
    (df['VDD004A']==6),
    # Superior completo ou equivalente
    (df['VDD004A']==7)
]

# Mapeamento de 'nivel_educacao':
# 1 = Médio incompleto/ não feito ou equivalente
# 2 = Médio Completo ou equivalente
# 3 = Superior Incompleto ou equivalente
# 4 = Superior completo ou equivalente
# -1 = Dados Inconsistentes (Default)
categorias = [1, 2, 3, 4]

df['nivel_educacao'] = np.select(condicoes, categorias, default=-1).astype(int)

print(df['nivel_educacao'].value_counts(dropna=False))

print(df.shape)

df.drop(inplace=True,columns=['VDD004A'])

(21084, 58)
nivel_educacao
1    9031
2    6754
4    4392
3     907
Name: count, dtype: int64
(21084, 59)


* As variáveis `N012`, `N016`, `N017` e `N018` se referem à frequência semanal de sintomas de saúde mental, e a variável `N00101` se refere à percepção geral de saúde do indivíduo.
---
* Será construído um indicador contínuo (Score de Saúde Mental) combinando os sintomas (peso de 75%) e a percepção geral (peso de 25%), com posterior discretização em 2 categorias com base no ponto de corte de 0.7.
* As categorias finais da discretização são:
    * Mau Score: `saude_mental_score` < 0.7 (Indica presença frequente de sintomas ou percepção de saúde debilitada)
    * Bom Score: `saude_mental_score` >= 0.7 (Indica ausência/baixa frequência de sintomas e boa percepção de saúde)* As variaveis `N00101`,`N016`,`N017` e `N018` se referem a autoperceção e a saúde mental.

In [37]:
print(df.shape)

# 1. Mapeamento das variáveis de sintomas (Invertendo a escala para que maior = melhor)
map_sintomas = {1: 1.0, 2: 0.75, 3: 0.5, 4: 0.25}
vars_sintomas = ['N012', 'N016', 'N017', 'N018']

for col in vars_sintomas:
    df[col] = df[col].map(map_sintomas)
    print(df[col].value_counts(dropna=False))

# 2. Mapeamento da percepção geral de saúde (N00101)
map_percepcao = {1: 1.0, 2: 0.8, 3: 0.6, 4: 0.4, 5: 0.2}
print(df['N00101'].value_counts(dropna=False))
df['N00101_normalizado'] = df['N00101'].map(map_percepcao)

print(df['N00101_normalizado'].value_counts(dropna=False))

# 3. Calcular a média estrita dos sintomas (ignora NaNs se houver alguma perda parcial)
df['media_sintomas'] = df[vars_sintomas].mean(axis=1)

print(df['media_sintomas'].value_counts(dropna=False))

# 4. Cálculo do Score de Saúde Mental Consolidado (Válido e Linear)
df['saude_mental_score'] = (df['media_sintomas'] * 0.75) + (df['N00101_normalizado'] * 0.25)

# Mapeamento de 'perfil_saude_mental':
# 1 = Bom Score (>= 0.80)
# 0 = Mau Score (< 0.80)
# 5. Aplicar a discretização numérica com base no corte de 0.80
df['perfil_saude_mental'] = np.where(
    df['saude_mental_score'] >= 0.80,
    1,
    0
).astype(int)

df.drop(inplace=True, columns=['media_sintomas', 'N00101_normalizado', 'N00101', 'V0001'] + vars_sintomas)

# Visualizar a distribuição do seu novo indicador preditivo numérico
print("Distribuição do Perfil de Saúde Mental:")
print(df['perfil_saude_mental'].value_counts().sort_index())

print(df.shape)

(21084, 58)
N012
1.00    16809
0.75     3214
0.50      572
0.25      489
Name: count, dtype: int64
N016
1.00    17156
0.75     2866
0.50      573
0.25      489
Name: count, dtype: int64
N017
1.00    18792
0.75     1632
0.50      349
0.25      311
Name: count, dtype: int64
N018
1.00    20538
0.75      375
0.50       98
0.25       73
Name: count, dtype: int64
N00101
2.0    13530
1.0     4541
3.0     2684
4.0      281
5.0       48
Name: count, dtype: int64
N00101_normalizado
0.8    13530
1.0     4541
0.6     2684
0.4      281
0.2       48
Name: count, dtype: int64
media_sintomas
1.0000    14908
0.9375     2643
0.8750     1400
0.8125      909
0.7500      425
0.6875      234
0.6250      198
0.5625      123
0.4375       94
0.5000       76
0.3125       28
0.3750       23
0.2500       23
Name: count, dtype: int64
Distribuição do Perfil de Saúde Mental:
perfil_saude_mental
0     1487
1    19597
Name: count, dtype: int64
(21084, 54)


## Renomeação das variaveis

In [38]:
# Renomeando uma ou mais colunas
df.rename(columns={
    'A01901': 'Acesso a Internet',
    'A016010': 'Acesso a Rede de Lixo',
    'D001':'Analfabeta',
    'VDF003': 'N° Salario min',
    'VDE014': 'Grupamento de trabalho principal',
    'C009': 'Cor ou Raça',
    'C006': 'Sexo',
    'C008': 'Idade',
    'Q074': 'Asma',
    'A009010': 'Nível de Acesso a agua potável'
}, inplace=True)

## Variáveis de Alimentação

* São as seguintes variáveis: `['P02401', 'P02601', 'P00901', 'P01101', 'P013', 'P015', 'P02001', 'P01601', 'P018', 'P02002', 'P023', 'P02501', 'P02602', 'P02801', 'P029', 'P01001', 'P006', 'P019']`.
* São ao todo **** variáveis.
* Será feito primeiro o entendimento dos valores vazios de cada variável.
---
### Dados Ausentes

* Para a variável `P01001`, a lógica de salto vem do conjunto `[P00901, P019]`, onde o respondente só preenche a pergunta caso o registro tenha um valor maior ou igual a 5.
* Como não é possível realizar uma imputação simples para os valores menores ou iguais a 4, será feita uma imputação por cluster, utilizando as variáveis de salto, dados de alimentação e características gerais.
    * Essa imputação ressignifica a váriavel, demosntrando quantas vezes por dia a pessoa come em média tal alimento.
* Já a variável `P02401` diz respeito ao tipo de consumo de leite. Se a pessoa não consome leite, a pergunta não é aplicável; logo, será imputado o valor 0 para representar essa categoria.


In [39]:
colunas_contexto = [
    'P018', 'P00901', 'P019', 'P01001',
    'P02602', 'P02501', 'P02001', 'P02601', 'P01101', 'P015', 'P006',
    'Sexo', 'nivel_educacao', 'regiao_br', 'IMC', 'Salario_min', 'Idade'
]
colunas_alvo = ['P019', 'P01001']

# Categoricos_ordinais = [P02601,nivel_educacao,IMC (está discretizado),salario_min (esta discrezado, por N° de salarios min)]
# Categoricos_nominais = [Sexo,regiao_br]

print(df['P019'].value_counts(dropna=False))

# --- Distribuição ANTES da imputação (para validação posterior) ---
dist_antes = {col: df[col].describe() for col in colunas_alvo}

# Isso cria colunas como 'Sexo_1', 'Sexo_2', 'regiao_br_1', 'regiao_br_2', etc.
df = pd.get_dummies(df, columns=['Sexo', 'regiao_br'])

# 2. Capturamos o nome dessas novas colunas binárias criadas
colunas_dummies = [col for col in df.columns if col.startswith(('Sexo_', 'regiao_br_'))]

# 3. Atualizamos a sua lista de colunas (removendo as antigas e adicionando as novas)
colunas_contexto = [
    'P018', 'P00901', 'P019', 'P01001',
    'P02602', 'P02501', 'P02001', 'P02601', 'P01101', 'P015', 'P006',
    'nivel_educacao', 'IMC', 'Salario_min', 'Idade'
] + colunas_dummies # <--- Somando a lista de dummies aqui

# --- Split estratificado ---
# Feito aqui e mantido para o resto da pipeline
df_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['Asma']
)
df_train = df_train.copy()
df_test  = df_test.copy()

# --- Scaler: fit apenas no treino ---
scaler = MinMaxScaler()
train_scaled = pd.DataFrame(
    scaler.fit_transform(df_train[colunas_contexto]),
    columns=colunas_contexto,
    index=df_train.index
)
test_scaled = pd.DataFrame(
    scaler.transform(df_test[colunas_contexto]),
    columns=colunas_contexto,
    index=df_test.index
)

# --- KNN: fit apenas no treino ---
# k=10 escolhido por ser mais estável que o default (5) dado o volume de dados (~18k treino)
# weights='distance' para que vizinhos mais próximos contribuam mais
imputer = KNNImputer(n_neighbors=11, weights='distance')
train_imputed_scaled = pd.DataFrame(
    imputer.fit_transform(train_scaled),
    columns=colunas_contexto,
    index=df_train.index
)
test_imputed_scaled = pd.DataFrame(
    imputer.transform(test_scaled),
    columns=colunas_contexto,
    index=df_test.index
)

# --- Reverter escala ---
train_original = pd.DataFrame(
    scaler.inverse_transform(train_imputed_scaled),
    columns=colunas_contexto,
    index=df_train.index
)
test_original = pd.DataFrame(
    scaler.inverse_transform(test_imputed_scaled),
    columns=colunas_contexto,
    index=df_test.index
)

# --- Devolver valores imputados aos splits (não ao df original) ---
for col in colunas_alvo:
    df_train[col] = train_original[col].round().astype(int)
    df_test[col]  = test_original[col].round().astype(int)

df_imputado_completo = pd.concat([df_train[colunas_alvo], df_test[colunas_alvo]])

# Atualiza as colunas no df original
df.update(df_imputado_completo)

# Validação rápida para garantir que não há mais nulos no df original
print(f"Nulos em P01001 no DF original: {df['P01001'].isna().sum()}")
print(f"Nulos em P019 no DF original: {df['P019'].isna().sum()}")

print(df['P019'].value_counts())

P019
NaN    12108
1.0     4885
2.0     3154
3.0      937
Name: count, dtype: int64
Nulos em P01001 no DF original: 0
Nulos em P019 no DF original: 0
P019
1.0    12484
2.0     7663
3.0      937
Name: count, dtype: int64


In [40]:
# ==========================================
# REVERTENDO OS DUMMIES PARA COLUNA ORIGINAL
# ==========================================

# 1. Revertendo a variável 'Sexo'
# Captura as colunas referentes ao Sexo (Sexo_1.0, Sexo_2.0, etc.)
cols_sexo = [col for col in df.columns if col.startswith('Sexo_')]

# Pega o nome da coluna que tem o valor máximo (1) na linha, apaga o prefixo e converte para número
df['Sexo'] = df[cols_sexo].idxmax(axis=1).str.replace('Sexo_', '').astype(float)

# Apaga as colunas dummies antigas
df = df.drop(columns=cols_sexo)


# 2. Revertendo a variável 'regiao_br'
# Captura as colunas referentes à região
cols_regiao = [col for col in df.columns if col.startswith('regiao_br_')]

# Pega o nome da coluna que tem o valor 1, apaga o prefixo e converte para inteiro
df['regiao_br'] = df[cols_regiao].idxmax(axis=1).str.replace('regiao_br_', '').astype(int)

# Apaga as colunas dummies antigas
df = df.drop(columns=cols_regiao)

# ==========================================
# Validação Rápida
# ==========================================
print("Variável Sexo restaurada:")
print(df['Sexo'].value_counts())

print("\nVariável Região restaurada:")
print(df['regiao_br'].value_counts())

Variável Sexo restaurada:
Sexo
1.0    10782
2.0    10302
Name: count, dtype: int64

Variável Região restaurada:
regiao_br
2    6932
3    4556
1    4259
4    2675
5    2662
Name: count, dtype: int64


## Arquitetura do Score de Qualidade Alimentar (Classificação NOVA)

Para criar um indicador robusto do perfil nutricional sem permitir que o maior número de perguntas sobre alimentos saudáveis gerasse um viés matemático, foi desenhada uma **Pipeline com Centralização por Grupos**.

### Componentes e Pesos Teóricos
1.  **Grupo 1 (In Natura / Minimamente Processados):** `P006`, `P01101`, `P013`, `P015`, `P023` + Variáveis compostas de densidade diária (`P0091` e `P0189`). **Peso do Grupo: +1.5**
2.  **Grupo 3 (Alimentos Processados):** `P01601` (Suco natural - considerando adição de açúcar/adoçante). **Peso do Grupo: -0.5**
3.  **Grupo 4 (Ultraprocessados):** `P02501`, `P02602` + variáveis de bebidas industriais (`P02001`, `P02002`) modificadas pelos seus respectivos tipos (`P02101`, `P02102`) onde a versão *Diet/Light* recebe penalidade atenuada. **Peso do Grupo: -1.5**

### Mecânica de Cálculo
* Todas as variáveis individuais são normalizadas para a escala `[0, 1]` dividindo pelo seu teto teórico.
* Calcula-se a **média aritmética interna** de cada um dos três grupos (gerando índices independentes de 0 a 1).
* **Equação Final:** $\text{Score} = (1.5 \times \text{Índice}_{G1}) - (0.5 \times \text{Índice}_{G3}) - (1.5 \times \text{Índice}_{G4})$
* **Intervalo Teórico do Score:** `[-2.0, +1.5]`

### Discretização Final (`perfil_alimentacao`)
O score contínuo foi segmentado em três faixas de corte fixas e interpretáveis:
* `1` = **Dieta Precária:** Score $< -0.5$ (Ultraprocessados dominam a rotina).
* `2` = **Dieta Balanceada:** Score entre $-0.5$ e $0.5$ (Consumo híbrido e equilibrado de comida real e industrializados).
* `3` = **Dieta Saudável:** Score $> 0.5$ (Forte predominância de alimentos *In Natura*).

In [41]:
print(df.shape)
pd.set_option('display.max_rows', None)
# --- PASSO 1: Engenharia de Recursos (Densidade de Consumo) ---
# Multiplica dias por vezes ao dia e divide por 7
df['P0091'] = (df['P00901'] * df['P01001']) / 7
df['P0189'] = (df['P018'] * df['P019']) / 7

# --- PASSO 2: Centralização do Grupo 1 (In Natura) ---
g1_puros = ['P006', 'P01101', 'P013', 'P015', 'P023']
g1_norm = pd.DataFrame(index=df.index)

# Normaliza variáveis de dias puros (0 a 7 -> 0 a 1)
for var in g1_puros:
    g1_norm[var] = df[var] / 7

# Normaliza as variáveis compostas (0 a 3 -> 0 a 1)
g1_norm['P0091'] = df['P0091'] / 3
g1_norm['P0189'] = df['P0189'] / 3

# Extrai a média do grupo (Índice de 0 a 1)
df['indice_g1'] = g1_norm.mean(axis=1)

# --- PASSO 3: Centralização do Grupo 3 (Processados) ---
df['indice_g3'] = df['P01601'] / 7

# --- PASSO 4: Centralização do Grupo 4 (Ultraprocessados com Modificadores) ---
# Dicionário mapeia o peso relativo de cada tipo dentro da penalidade do grupo
fator_tipo = {
    1: (1.0 / 1.5),   # Diet/Light/Zero atenua para -1.0
    2: 1.0,           # Normal mantém penalidade cheia -1.5
    3: (1.25 / 1.5)   # Ambos fica no intermediário -1.25
}

# Cria os modificadores (caso seja nulo devido ao pulo, assume peso padrão 1.0)
mod_suco = df['P02101'].map(fator_tipo).fillna(1.0)
mod_refri = df['P02102'].map(fator_tipo).fillna(1.0)

g4_norm = pd.DataFrame(index=df.index)
g4_norm['P02501'] = df['P02501'] / 7
g4_norm['P02602'] = df['P02602'] / 7
g4_norm['P02001'] = (df['P02001'] / 7) * mod_suco
g4_norm['P02002'] = (df['P02002'] / 7) * mod_refri

# Extrai a média do grupo (Índice de 0 a 1)
df['indice_g4'] = g4_norm.mean(axis=1)

# --- PASSO 5: Cálculo do Score Final Ponderado ---
# O intervalo resultante vai de -2.0 a +1.5
df['score_nova'] = (1.5 * df['indice_g1']) - (0.5 * df['indice_g3']) - (1.5 * df['indice_g4'])

# --- PASSO 6: Discretização em Categorias Numéricas ---
condicoes_score = [
    (df['score_nova'] < -0.25),
    (df['score_nova'].between(-0.25, 0.25)),
    (df['score_nova'] > 0.25)
]

# Mapeamento de 'perfil_alimentacao':
# 1 = Dieta precária (< -0.5)
# 2 = Dieta balanceada (entre -0.5 e 0.5)
# 3 = Dieta saudável (> 0.5)
categorias_score = [1, 2, 3]

df['perfil_alimentacao'] = np.select(condicoes_score, categorias_score, default=2).astype(int)

# --- PASSO 7: Limpeza do Dataset ---
# Remove as colunas originais de alimentação e os índices intermediários criados
colunas_para_remover = [
    'P006', 'P00901', 'P01001', 'P01101', 'P013', 'P015', 'P01601', 'P018', 'P019',
    'P02001', 'P02002', 'P02101', 'P02102', 'P023', 'P02401', 'P02501', 'P02601', 'P02602',
    'P0091', 'P0189', 'indice_g1', 'indice_g3', 'indice_g4'
]

df.drop(inplace=True, columns=colunas_para_remover+['Unnamed: 0'])

# Visualizar resultados
print("Distribuição do perfil de alimentação:")
print(df['perfil_alimentacao'].value_counts().sort_index())
print(df.shape)

(21084, 54)
Distribuição do perfil de alimentação:
perfil_alimentacao
1     1501
2    10647
3     8936
Name: count, dtype: int64
(21084, 37)


* As variaveis `P027`,`P02801`,`P029` se referem ao consumo de alcool em média.
---
* `P027` é se a pessoa consome alcool, caso não consuma (==1) as proximas variaveis são vazias.
* `P02801` só é respondido quando `P027` == 3, e é a quantidade de dias da semana que a pessoa consume alcool.
* `P029` é quantas doses a pessoa toma quando bebe.
* Como `P027` N° de vezes que se consume em Mês, caso a pessoa responda 2 (1 vez ou menos), a resposta de `P029` será divida por 4 (transformado media semanal).
* Para o registro que Responder `P02801` o N° de dias será multiplicado pelo numero de doses e divido por 7 (media semanal)

In [42]:
print(df.shape)

# 1. Tratar valores nulos iniciais para evitar quebras lógicas
df['P02801'] = df['P02801'].fillna(0)
df['P029'] = df['P029'].fillna(0)

# 2. Definir as condições para estimar os Dias Equivalentes por Semana
condicoes_alcool = [
    (df['P027'] == 1),                               # Não bebe nunca -> 0 dias
    (df['P027'] == 2),                               # Menos de uma vez por mês -> 0.125 dias
    (df['P027'] == 3) & (df['P02801'] == 0),         # Uma vez ou mais por mês, mas < 1x por semana -> 0.5 dias
    (df['P027'] == 3) & (df['P02801'] >= 1)          # Consumo semanal fixo -> Usa o próprio número de dias relatado
]

valores_dias = [
    0.0,
    0.125,
    0.5,
    df['P02801']
]

# Criar a variável intermediária de equivalência temporal
df['dias_equivalentes_semana'] = np.select(condicoes_alcool, valores_dias, default=0.0)

# 3. Calcular o indicador contínuo de volume (Total de Doses Semanais)
df['doses_semanais_alcool'] = df['dias_equivalentes_semana'] * df['P029']

# 4. Discretização Numérica em Categorias de Risco
condicoes_discreta = [
    (df['doses_semanais_alcool'] == 0),                                      # Nenhuma dose
    (df['doses_semanais_alcool'] > 0) & (df['doses_semanais_alcool'] <= 7),  # Consumo Leve a Moderado (Até 1 dose/dia em média)
    (df['doses_semanais_alcool'] > 7)                                        # Consumo de Risco / Elevado (Mais de 1 dose/dia em média)
]

# Mapeamento de 'perfil_alcool':
# 1 = Abstêmio / Não consome
# 2 = Consumo Moderado
# 3 = Consumo de Risco
categorias_alcool = [1, 2, 3]

df['perfil_alcool'] = np.select(condicoes_discreta, categorias_alcool, default=1).astype(int)

# 5. Limpeza do dataset (remover variáveis originais e suportes intermediários)
colunas_para_remover = ['P027', 'P02801', 'P029', 'dias_equivalentes_semana', 'doses_semanais_alcool']
df.drop(inplace=True, columns=colunas_para_remover)

# Visualizar a distribuição do novo indicador discreto
print("Distribuição do perfil de consumo de álcool:")
print(df['perfil_alcool'].value_counts().sort_index())

print(df.shape)

(21084, 37)
Distribuição do perfil de consumo de álcool:
perfil_alcool
1    11517
2     6862
3     2705
Name: count, dtype: int64
(21084, 35)


* A variavel `perfil_exercicio` se refere ao perfil de prática de atividade física ou esporte do registro.
---
* Ele possui inicialmente dados distribuídos em 4 variáveis originais com lógicas de salto (`P034`, `P035`, `P03701`, `P03702`), sendo feita uma engenharia de recursos para consolidar o volume em minutos semanais e discretizar o resultado em 3 categorias.
* As categorias são:
    * Inativo: `minutos_semanais_exercicio == 0` (Mapeado como Código 1)
        * *Justificativa:* Agrupa os respondentes que declararam não ter praticado exercícios nos últimos três meses (`P034 == 2`) ou que informaram uma frequência de dias nula (`P035 == 0`), representando a ausência dessa prática regular no estilo de vida.
    * Insuficientemente Ativo: `0 < minutos_semanais_exercicio < 150` (Mapeado como Código 2)
        * *Justificativa:* Mapeia os indivíduos que possuem o hábito de se exercitar ao longo da semana, mas cujo volume total ponderado de tempo fica abaixo do limiar mínimo ideal de proteção.
    * Ativo: `minutos_semanais_exercicio >= 150` (Mapeado como Código 3)
        * *Justificativa:* Consolida quem atinge ou supera a diretriz oficial da Organização Mundial da Saúde (OMS) de pelo menos 150 minutos de atividade física semanal moderada a intensa, agindo como um marcador direto de proteção metabólica e redução de risco para doenças crônicas.
* As variáveis de suporte e de tempo bruto são descartadas ao final do processo para garantir o correto balanceamento e a não-colinearidade nos algoritmos.

In [43]:
print(df.shape)

# 1. Tratar valores nulos para evitar quebras matemáticas nas regras de salto
df['P035'] = df['P035'].fillna(0)
df['P03701'] = df['P03701'].fillna(0)
df['P03702'] = df['P03702'].fillna(0)

# 2. Calcular o volume total de minutos semanais praticados
# Fórmula: Dias por semana * (Horas * 60 + Minutos)
df['minutos_semanais_exercicio'] = df['P035'] * ((df['P03701'] * 60) + df['P03702'])

# Forçar zero para quem respondeu "Não" na pergunta de triagem (P034 == 2)
df.loc[df['P034'] == 2, 'minutos_semanais_exercicio'] = 0.0

# 3. Discretização Numérica Baseada na Diretriz da OMS (150 minutos)
condicoes_exercicio = [
    (df['minutos_semanais_exercicio'] == 0),                                       # Não pratica
    (df['minutos_semanais_exercicio'] > 0) & (df['minutos_semanais_exercicio'] < 150), # Pratica, mas não bate a meta
    (df['minutos_semanais_exercicio'] >= 150)                                      # Ativo (Atinge a recomendação da OMS)
]

# Mapeamento de 'perfil_exercicio':
# 1 = Inativo
# 2 = Insuficientemente Ativo
# 3 = Ativo
categorias_exercicio = [1, 2, 3]

# Correção feita aqui: mudado de categories_exercicio para categorias_exercicio
df['perfil_exercicio'] = np.select(condicoes_exercicio, categorias_exercicio, default=1).astype(int)

# 4. Limpeza do dataset (Remover colunas de suporte e originais)
colunas_para_remover = ['P034', 'P035', 'P03701', 'P03702', 'minutos_semanais_exercicio']
df.drop(inplace=True, columns=colunas_para_remover)

# Visualizar a distribuição do novo indicador discreto
print("Distribuição do perfil de exercício físico:")
print(df['perfil_exercicio'].value_counts().sort_index())

print(df.shape)

(21084, 35)
Distribuição do perfil de exercício físico:
perfil_exercicio
1    12156
2     2906
3     6022
Name: count, dtype: int64
(21084, 32)


* A variavel `perfil_tempo_tela` se refere ao tempo médio diário de exposição a telas de lazer (Televisão e Computador/Tablet/Celular) do registro.
---
* Ele possui inicialmente dados distribuídos em 2 variáveis categóricas de faixas de tempo (`P04501` e `P04502`), sendo feita uma engenharia de recursos para converter as faixas nos seus pontos médios em horas, calcular a média simples do tempo de tela diário e discretizar o resultado final em 4 categorias.
* As categorias são:
    * Não assisto/uso: `horas_tv == 0` e `horas_telas == 0` (Mapeado como Código 1)
        * *Justificativa:* Isola os respondentes que declararam absenteísmo total para ambos os hábitos de tela diários (marcaram opção 6 tanto para TV quanto para dispositivos eletrônicos de lazer).
    * < 1 h: `0 < media_tempo_tela < 1.0` (Mapeado como Código 2)
        * *Justificativa:* Filtra os indivíduos com hábitos de exposição residual ou muito leve a mídias visuais no seu tempo livre.
    * 1 a 3 h: `1.0 <= media_tempo_tela <= 3.0` (Mapeado como Código 3)
        * *Justificativa:* Enquadra o perfil padrão de consumo moderado de entretenimento digital diário.
    * 3h +: `media_tempo_tela > 3.0` (Mapeado como Código 4)
        * *Justificativa:* Identifica o perfil de alto comportamento sedentário digital. A literatura médica associa faixas superiores a 3 horas diárias de tela recreativa a maiores prevalências de distúrbios de sono, ansiedade e riscos metabólicos crônicos.
* As variáveis categóricas de faixas brutas e os cálculos horários intermediários são dropados ao fim do processo para manter o dataset limpo e otimizado.

In [44]:
print(df.shape)


# 2. Converter os intervalos categóricos em horas contínuas (ponto médio do intervalo)
# 1 (<1h) -> 0.5h | 2 (1h a 2h) -> 1.5h | 3 (2h a 3h) -> 2.5h
# 4 (3h a 6h) -> 4.5h | 5 (6h+) -> 6.0h | 6 (Não assisto/uso) -> 0.0h
mapeamento_horas = {1: 0.5, 2: 1.5, 3: 2.5, 4: 4.5, 5: 6.0, 6: 0.0}

df['horas_tv'] = df['P04501'].map(mapeamento_horas)
df['horas_telas'] = df['P04502'].map(mapeamento_horas)

# Calcular a média simples do tempo de exposição diária
df['media_tempo_tela'] = (df['horas_tv'] + df['horas_telas']) / 2

# 3. Definir as condições de discretização solicitadas
condicoes_tela = [
    (df['horas_tv'] == 0.0) & (df['horas_telas'] == 0.0),                     # Não assisto/uso ambos
    (df['media_tempo_tela'] > 0.0) & (df['media_tempo_tela'] < 1.0),          # < 1 h
    (df['media_tempo_tela'] >= 1.0) & (df['media_tempo_tela'] <= 3.0),        # 1 a 3 h
    (df['media_tempo_tela'] > 3.0)                                            # 3h +
]

# Mapeamento numérico do perfil:
# 1 = Não assisto/uso
# 2 = < 1 h
# 3 = 1 a 3 h
# 4 = 3h +
categorias_tela = [1, 2, 3, 4]

df['perfil_tempo_tela'] = np.select(condicoes_tela, categorias_tela, default=1).astype(int)

# 4. Limpeza do dataset (Remover colunas originais e suportes temporários)
colunas_para_remover = ['P04501', 'P04502', 'horas_tv', 'horas_telas', 'media_tempo_tela']
df.drop(inplace=True, columns=colunas_para_remover)

# Visualizar a distribuição resultante
print("Distribuição do perfil de tempo de tela:")
print(df['perfil_tempo_tela'].value_counts(dropna=False).sort_index())

print(df.shape)

(21084, 32)
Distribuição do perfil de tempo de tela:
perfil_tempo_tela
1      543
2     4391
3    12874
4     3276
Name: count, dtype: int64
(21084, 31)


In [45]:
df_copy = df.copy()

df_copy.to_csv(r'../Arquivos Csv/Etapa5.csv')